# Лаборатори 4: Нэгдүгээр эрэмбийн логик
Prolog ба pyDatalog ашиглан гэр бүлийн мэдлэгийн сан байгуулах

## 1. Зорилго
Энэхүү лабораториор нэгдүгээр эрэмбийн логикийн баримт (fact), дүрэм (rule) болон асуулт (query)-ыг практикт ашиглаж сурсан. pyDatalog болон Prolog ашиглан гэр бүлийн харилцааны мэдлэгийн сан үүсгэж, түүнээс логик дүгнэлтээр хэрэгтэй мэдээллүүдээ шүүж авах чадвартай болох зорилготой.

## 2. Онол
Нэгдүгээр эрэмбийн логикт орчныг объект болон тэдгээрийн хоорондох харилцаагаар илэрхийлдэг. Гол ойлголтууд:
- **Баримт (Fact):** Шууд өгөгдсөн үнэн мэдээлэл (жишээ нь: `parent('bataa', 'bold')`).
- **Дүрэм (Rule):** Хэрэв нөхцөл биелэвэл шинэ баримт гаргах өгүүлбэр (Horn clause хэлбэртэй).
- **Unification:** Асуултын хувьсагчийг мэдлэгийн санд байгаа тодорхой утгатай тохируулан харгалзуулах үйл явц.
- **Closed-World Assumption:** Мэдлэгийн санд байхгүй эсвэл батлагдахгүй зүйлийг шууд 'худал' гэж үзэх зарчим.

## 3. Аргачлал

### Алхам 1: Орчин бэлтгэх

In [ ]:
# !pip install pyDatalog
from pyDatalog import pyDatalog
pyDatalog.clear()

# Туршилтын жишээ
pyDatalog.create_terms('bird, flies, X')
+bird('tweety')
+bird('sparrow')
flies(X) <= bird(X)
print(flies(X))

In [ ]:
pyDatalog.clear()

# Шаардлагатай предикатуудыг зарлах
pyDatalog.create_terms('male, female, parent, person, has_child, childless, '
                       'father, mother, son, daughter, grandparent, sibling, brother, '
                       'sister, uncle, aunt, cousin, ancestor, descendant, '
                       'great_grandparent, sibling_test, spouse, uncle_in_law, aunt_in_law, '
                       'X, Y, Z, P, Q')

# Туслах функцүүд
def count(q):
    r = pyDatalog.ask(q)
    return 0 if r is None else len(r.answers)

def names(q):
    r = pyDatalog.ask(q)
    return [] if r is None else sorted(a[0] for a in r.answers)

### Алхам 2: Гэр бүлийн мэдлэгийн сан (баримтууд)

In [ ]:
# Хүйсийн баримтууд
for m in ['bataa', 'bold', 'tomor', 'dorj', 'zolboo', 'munkh', 'temuulen', 'bat']:
    +male(m)
for f in ['saraa', 'oyuna', 'tsetseg', 'ganaa', 'anar', 'naraa', 'enkhee']:
    +female(f)

# Эцэг эх - хүүхдийн баримтууд
families = [
    (('bataa', 'saraa'),  ['bold', 'oyuna', 'tomor']),
    (('bold', 'tsetseg'), ['anar', 'zolboo']),
    (('dorj', 'oyuna'),   ['munkh', 'naraa']),
    (('tomor', 'ganaa'),  ['enkhee']),
    (('bat', 'anar'),     ['temuulen']),
]
for pair, kids in families:
    for p in pair:
        for c in kids:
            +parent(p, c)

print('parent:', count("parent(X, Y)"), ' male:', count("male(X)"), ' female:', count("female(X)"))
print("parent('bataa', X):", names("parent('bataa', X)"))

### Алхам 3: Энгийн дүрмүүд

In [ ]:
# father, mother, son, daughter дүрмүүд
father(X, Y) <= parent(X, Y) & male(X)
mother(X, Y) <= parent(X, Y) & female(X)
son(X, Y) <= parent(Y, X) & male(X)
daughter(X, Y) <= parent(Y, X) & female(X)

# person дүрэм
person(X) <= male(X)
person(X) <= female(X)

print("father(X, 'anar'):  ", names("father(X, 'anar')"))
print("mother(X, 'bold'):  ", names("mother(X, 'bold')"))
print("son(X, 'oyuna'):    ", names("son(X, 'oyuna')"))
print("daughter(X, 'oyuna'):", names("daughter(X, 'oyuna')"))
print('father:', count("father(X, Y)"), ' mother:', count("mother(X, Y)"), ' person:', count("person(X)"))

### Алхам 4: Өвөө эмээ ба ах дүү

In [ ]:
# grandparent болон sibling дүрмүүд
grandparent(X, Y) <= parent(X, Z) & parent(Z, Y)
sibling(X, Y) <= parent(Z, X) & parent(Z, Y) & (X != Y)

print("grandparent(X, 'temuulen'):", names("grandparent(X, 'temuulen')"))
print("sibling(X, 'munkh'):", names("sibling(X, 'munkh')"))
print("sibling(X, 'bold'): ", names("sibling(X, 'bold')"))

### Алхам 5: Ураг төрлийн дүрмүүд

In [ ]:
brother(X, Y) <= sibling(X, Y) & male(X)
sister(X, Y) <= sibling(X, Y) & female(X)
uncle(X, Y) <= parent(P, Y) & brother(X, P)
aunt(X, Y) <= parent(P, Y) & sister(X, P)
cousin(X, Y) <= parent(P, X) & parent(Q, Y) & sibling(P, Q)

print("brother(X, 'oyuna'):  ", names("brother(X, 'oyuna')"))
print("uncle(X, 'anar'):     ", names("uncle(X, 'anar')"))
print("cousin(X, 'enkhee'):  ", names("cousin(X, 'enkhee')"))

### Алхам 6: Рекурсив дүрэм

In [ ]:
# Рекурсив ancestor дүрэм
ancestor(X, Y) <= parent(X, Y)
ancestor(X, Y) <= parent(X, Z) & ancestor(Z, Y)
descendant(X, Y) <= ancestor(Y, X)

print("ancestor(X, 'temuulen'):", names("ancestor(X, 'temuulen')"))
print("descendant(X, 'bataa'): ", names("descendant(X, 'bataa')"))

### Алхам 7: Үгүйсгэл ба асуултууд

In [ ]:
has_child(X) <= parent(X, Y)
childless(X) <= person(X) & ~has_child(X)
great_grandparent(X, Y) <= parent(X, Z) & grandparent(Z, Y)

print('childless:', names("childless(X)"))
print("great_grandparent(X, 'temuulen'):", names("great_grandparent(X, 'temuulen')"))

## 4. Үр дүн

### Асуултуудын хүснэгт

| № | Асуулт | Код / Дүрэм | Гаргаж авсан хариулт |
|---|---|---|---|
| 1 | Болдын хүүхдүүд | `parent('bold', X)` | anar, zolboo |
| 2 | Temuulen-ийн өвөө эмээ | `grandparent(X, 'temuulen')` | bold, tsetseg |
| 3 | Munkh-ийн авга ах нар | `uncle(X, 'munkh')` | bold, tomor |
| 4 | Enkhee-ийн үеэлүүд | `cousin(X, 'enkhee')` | anar, munkh, naraa, zolboo |
| 5 | Үеэлтэй хүмүүс | `cousin(X, Y)` | 5 хүн: anar, enkhee, munkh, naraa, zolboo |
| 6 | Хүүхэдгүй хүмүүс | `childless(X)` | enkhee, munkh, naraa, temuulen, zolboo |
| 7 | Temuulen-ийн эцэг эхийн өвөө эмээ | `great_grandparent(X, 'temuulen')` | bataa, saraa |
| 8 | Ач, зээтэй хүмүүс | `grandparent(X, Y)` | bataa, bold, saraa, tsetseg |

### Асуултуудын товч хариулт:
- **Алхам 4 ($X \neq Y$-гүй үед):** sibling-ийн тоо 10-аас 19 болж нэмэгдсэн. Шалтгаан нь эцэг эхтэй 9 хүн тус бүр өөрөө өөрийнхөө ах дүү болох `sibling(X, X)` мөрүүд давхар гарч ирсэн.
- **Алхам 8 (Prolog vs pyDatalog):** Prolog нь хайлтын бүх замыг шалгадаг тул `sibling(oyuna, Y)` дээр Bataa, Saraa гэсэн 2 эцэг эхийн замаар давхардан 4 хариу буцаадаг. pyDatalog нь Datalog дээр суурилсан тул хариуг олонлог хэлбэрээр автоматаар давхардалгүй (2 хариу) харуулдаг.
- **Нэмэлт даалгавар 1 (spouse):** `spouse` дүрэм нь нийтлэг хүүхдээр нь хосыг тодорхойлдог. Хүүхэдгүй хосыг олохын тулд `married('bataa', 'saraa')` гэх мэт шууд гэрлэлтийн баримт нэмэх шаардлагатай.

## 6. Дүгнэлт

Энэхүү лабораториор логик програpoint-ийн давуу талыг практикт туршиж үзлээ. Ердөө хэдхэн суурь баримт (`male`, `female`, `parent`) оруулахад дүрмүүд ашиглан ураг төрлийн нарийн холбоонуудыг олон давхар алгоритм бичихгүйгээр шууд логик дүгнэлтээр гаргаж авах боломжтойг харлаа.

Мөн Closed-world assumption болон unsafe negation-ийн талаар ойлголттой болж, `childless` дүрэм дээр `person(X)`-ийг эхэлж холбох шаардлагатайг туршиж үзлээ. Рекурсив дүрэм нь олон үе дамжсан харилцааг хялбар тодорхойлдог бөгөөд pyDatalog ба Prolog-ийн үр дүн боловсруулах ялгааг амжилттай харьцуулж сурлаа.